# 10. External testing integrations

Advanced maintainer notebook for Rhesis and Semantica evaluation. This is optional and not part of the required live 3-hour participant flow.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiment' / 'run.py').exists():
    candidates = [Path('/content/kdd'), Path('/content/drive/MyDrive/kdd'), Path('/Users/syaikhipin/Documents/Phd/Publish Paper/kdd')]
    PROJECT_ROOT = next((p for p in candidates if (p / 'experiment' / 'run.py').exists()), PROJECT_ROOT)

EXPERIMENT_DIR = PROJECT_ROOT / 'experiment'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
sys.path.insert(0, str(EXPERIMENT_DIR))
print('PROJECT_ROOT =', PROJECT_ROOT)

## Integration availability

Rhesis requires `rhesis-sdk` and `RHESIS_API_KEY` in the environment. Semantica requires the `semantica` package. The notebook only checks presence by default.

In [ ]:
availability = {
    'rhesis_sdk_importable': importlib.util.find_spec('rhesis') is not None,
    'rhesis_api_key_present': bool(os.environ.get('RHESIS_API_KEY')),
    'semantica_importable': importlib.util.find_spec('semantica') is not None,
}
print(json.dumps(availability, indent=2))

## Offline semantic evaluation

This deterministic path is safe for CI, Colab, and tutorial participants.

In [ ]:
cmd = [
    sys.executable, str(EXPERIMENT_DIR / 'run.py'),
    '--mode', 'real',
    '--backend', 'offline',
    '--datasets', 'locomo', 'longmemeval', 'memoryarena',
    '--longmemeval-files', 'longmemeval_oracle.json',
    '--max-conversations', '2',
    '--max-questions', '20',
    '--max-items', '100',
    '--top-k', '5',
    '--eval-backend', 'offline',
    '--eval-limit', '50',
]
print(' '.join(map(str, cmd)))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=240)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0

## Optional Rhesis smoke test

Set `RUN_RHESIS_SMOKE=1` and provide `RHESIS_API_KEY` outside the notebook to run this.

In [ ]:
if os.environ.get('RUN_RHESIS_SMOKE') == '1':
    cmd = [sys.executable, str(EXPERIMENT_DIR / 'run.py'), '--mode', 'real', '--eval-backend', 'rhesis', '--eval-smoke-test', '--eval-limit', '1']
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=240)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print('Skipped Rhesis smoke test.')

## Optional Semantica smoke test

In [ ]:
if availability['semantica_importable']:
    cmd = [sys.executable, str(EXPERIMENT_DIR / 'run.py'), '--mode', 'real', '--eval-backend', 'semantica', '--eval-smoke-test', '--eval-limit', '1']
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=240)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print('Skipped Semantica smoke test because semantica is not importable.')

## Optional Modal GPU runner

Use this only for maintainer or paper-quality runs. Keep Modal credentials in the environment and set `RUN_MODAL_SMOKE=1` to execute a tiny remote GPU smoke test.

In [ ]:
if os.environ.get('RUN_MODAL_SMOKE') == '1':
    cmd = [
        sys.executable, str(EXPERIMENT_DIR / 'run.py'),
        '--runner', 'modal',
        '--modal-gpu', os.environ.get('MODAL_GPU', 'T4'),
        '--modal-detach',
        '--mode', 'real',
        '--datasets', 'locomo',
        '--max-conversations', '1',
        '--max-questions', '1',
        '--eval-backend', 'offline',
        '--eval-smoke-test',
        '--visualize',
    ]
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=3600)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print('Skipped Modal GPU smoke test.')
    print('Detached Modal runs print a call id; fetch with --modal-call-id <call-id>.')

In [ ]:
latest = sorted(RESULTS_DIR.glob('run_*_real_metrics.json'))[-1]
metrics = json.loads(latest.read_text())['real']
for dataset, summary in metrics['datasets'].items():
    for strategy, row in summary['by_strategy'].items():
        if 'mean_semantic_score' in row:
            print(dataset, strategy, row['semantic_coverage_rate'], row['mean_semantic_score'], row['semantic_pass_rate'])